1. Parse the raw JSON out of Bronze:

In [0]:
import ast
from pyspark.sql import Row

bronze_df = spark.table("retail_demo.retail_bronze.weather_raw")
bronze_rows = bronze_df.collect()

parsed_rows = []
for r in bronze_rows:
    try:
        payload = ast.literal_eval(r.raw_response)
        current = payload.get("current", {})
        parsed_rows.append(Row(
            store_id=r.store_id,
            city=r.city,
            region=r.region,
            observation_time=current.get("time"),
            temperature_f=float(current.get("temperature_2m")) if current.get("temperature_2m") is not None else None,
            precipitation_in=float(current.get("precipitation")) if current.get("precipitation") is not None else None,
            weather_code=current.get("weather_code"),
            wind_speed_mph=float(current.get("wind_speed_10m")) if current.get("wind_speed_10m") is not None else None,
            humidity_pct=float(current.get("relative_humidity_2m")) if current.get("relative_humidity_2m") is not None else None,
            ingestion_timestamp=r.ingestion_timestamp,
        ))
    except Exception as e:
        print(f"Failed to parse row for {r.store_id}: {e}")

silver_df = spark.createDataFrame(parsed_rows)
display(silver_df)

2. Basic data quality checks — worth running every time, and worth being able to talk about:

In [0]:
from pyspark.sql.functions import col

print("Row count:", silver_df.count())
print("Null temperature readings:", silver_df.filter(col("temperature_f").isNull()).count())
print("Distinct stores:", silver_df.select("store_id").distinct().count())

3. Deduplicate — keep latest ingestion per store/observation time:

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("store_id", "observation_time").orderBy(desc("ingestion_timestamp"))

deduped_df = silver_df.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

4. MERGE into the Silver Delta table:

In [0]:
# Create schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_demo.retail_silver")

In [0]:
if not spark.catalog.tableExists("retail_demo.retail_silver.weather_clean"):
    deduped_df.write.format("delta").saveAsTable("retail_demo.retail_silver.weather_clean")
    print("Created Silver table")
else:
    deduped_df.createOrReplaceTempView("silver_updates")
    spark.sql("""
        MERGE INTO retail_demo.retail_silver.weather_clean AS target
        USING silver_updates AS source
        ON target.store_id = source.store_id AND target.observation_time = source.observation_time
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("Merged into Silver table")

5. Sanity check:

In [0]:
%sql
SELECT store_id, city, observation_time, temperature_f, precipitation_in
FROM retail_demo.retail_silver.weather_clean
ORDER BY store_id;

6. New parsing block for the forecast

In [0]:
import ast
from pyspark.sql import Row

bronze_df = spark.table("retail_demo.retail_bronze.weather_raw")
bronze_rows = bronze_df.collect()

forecast_rows = []
for r in bronze_rows:
    try:
        payload = ast.literal_eval(r.raw_response)
        daily = payload.get("daily", {})
        dates = daily.get("time", [])
        for i, forecast_date in enumerate(dates):
            forecast_rows.append(Row(
                store_id=r.store_id,
                city=r.city,
                region=r.region,
                forecast_date=forecast_date,
                temp_max_f=daily.get("temperature_2m_max", [None]*len(dates))[i],
                temp_min_f=daily.get("temperature_2m_min", [None]*len(dates))[i],
                precip_sum_in=daily.get("precipitation_sum", [None]*len(dates))[i],
                precip_prob_pct=daily.get("precipitation_probability_max", [None]*len(dates))[i],
                wind_max_mph=daily.get("wind_speed_10m_max", [None]*len(dates))[i],
                ingestion_timestamp=r.ingestion_timestamp,
            ))
    except Exception as e:
        print(f"Failed to parse forecast for {r.store_id}: {e}")

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

forecast_schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("region", StringType(), True),
    StructField("forecast_date", StringType(), True),
    StructField("temp_max_f", DoubleType(), True),
    StructField("temp_min_f", DoubleType(), True),
    StructField("precip_sum_in", DoubleType(), True),
    StructField("precip_prob_pct", IntegerType(), True),
    StructField("wind_max_mph", DoubleType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])

forecast_df = spark.createDataFrame(forecast_rows, schema=forecast_schema)

# Dedup: keep the most recently ingested forecast per store/date
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, col

w = Window.partitionBy("store_id", "forecast_date").orderBy(desc("ingestion_timestamp"))
deduped_forecast_df = forecast_df.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

if not spark.catalog.tableExists("retail_demo.retail_silver.weather_forecast_clean"):
    deduped_forecast_df.write.format("delta").saveAsTable("retail_demo.retail_silver.weather_forecast_clean")
else:
    deduped_forecast_df.createOrReplaceTempView("forecast_updates")
    spark.sql("""
        MERGE INTO retail_demo.retail_silver.weather_forecast_clean AS target
        USING forecast_updates AS source
        ON target.store_id = source.store_id AND target.forecast_date = source.forecast_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)